In [8]:
# Import required libraries -- 00
import pennylane as qml
from pennylane import numpy as np
import plotly.express as plot

In [9]:
# Create noise simulator -- 02
import qiskit_aer.noise as noise
from qiskit_ibm_runtime.fake_provider import FakeAthensV2 as QCPU
noise_model = noise.NoiseModel.from_backend(QCPU())

In [10]:
# Assign simulator parameters -- 02
wires = [0, 1]
shots  = None or 500
dev = qml.device('qiskit.aer', wires=wires, noise_model=noise_model)
np.set_printoptions(formatter={'all': lambda x: f"{x:.3f}" if np.iscomplexobj(x) or np.isrealobj(x) else str(x)})

In [11]:
# Cooler print format -- 00
def probs_to_bits(probs, args:list=[], mode : str = 'bits'):
    from itertools import product
    for arg in args:
        if arg not in probs:
            continue
        if mode == 'bits':
            probs[arg] = {str(''.join(map(str, id))): np.float64(probs[arg][i]) for i, id in enumerate(product([0, 1], repeat=len(wires)))}
        if mode == 'int':
            probs[arg] = { i : np.float64(probs[arg][i]) for i in range(0, len(probs[arg]))}
    return probs

In [12]:
# Circuit declaration and display -- 01
@qml.qnode(dev, shots=shots)
def circuit(params = []):
    qml.Hadamard(0)
    qml.CNOT([0,1])
    return {
        'probs': qml.probs([0, 1]),
        'shots': qml.sample() if shots else []
    }
print(qml.draw(circuit)())

0: ──H─╭●─┤ ╭Probs  Sample
1: ────╰X─┤ ╰Probs  Sample


In [13]:
# Execute circuit -- 00
data = probs_to_bits(circuit(), ['probs'])
data

{'probs': {'00': np.float64(0.478),
  '01': np.float64(0.014),
  '10': np.float64(0.016),
  '11': np.float64(0.492)},
 'shots': array([[1.000, 1.000],
        [0.000, 0.000],
        [0.000, 0.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [0.000, 1.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [0.000, 0.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [1.000, 1.000],
        [0.000, 0.000],
        [

In [14]:
# Result analisys and graph rendering -- 00
graph = plot.bar(x=data['probs'].keys(), y=data['probs'].values(), labels={"x": "Sequence", "y": "Occurrence"}, range_y=[0,1], height=500, width=800, text_auto=True)
graph.show()